# Session 2: real power, reactive power and power factor
**A. Real-world problem.** A workshop pulls 100 kW from a 400 V supply, but its motors make the
current lag the voltage. The supply cable must carry far more current than 100 kW / 400 V
would suggest, and the utility charges a penalty for a low power factor. A bank of capacitors
fixes it. How big must the bank be, and why does high-voltage transmission lose less power?

**Plan.** (1) the power triangle, (2) current at a given power factor, (3) capacitor
correction, (4) line loss I^2 R at two voltages, (5) a design problem.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the power triangle
Voltage V and current I (both RMS) are sine waves, with the current lagging by the angle phi.
  real power      P = V I cos(phi)    (watts, does the useful work)
  reactive power  Q = V I sin(phi)    (var, swaps back and forth with motor/coil fields)
  apparent power  S = V I             (volt-amperes, what the wires must carry)
They form a right triangle: S^2 = P^2 + Q^2, and the power factor pf = cos(phi) = P / S.

In [ ]:
def power_triangle(v_rms, i_rms, phi_deg):
    """Return (P, Q, S, pf) for RMS voltage (V), RMS current (A) and lag angle (degrees)."""
    phi = np.radians(phi_deg)
    s = v_rms * i_rms
    return s * np.cos(phi), s * np.sin(phi), s, float(np.cos(phi))


def current_for(p_watts, v_rms, pf):
    """RMS current (A) a single-phase load needs for real power P at power factor pf: I = P / (V pf)."""
    return p_watts / (v_rms * pf)


def reactive_for(p_watts, pf):
    """Reactive power Q (var) of a load with real power P and power factor pf (lagging)."""
    return p_watts * np.tan(np.arccos(pf))

## Step 2: correcting with a capacitor
A capacitor supplies reactive power locally, so the supply does not have to. To lift the power
factor from pf1 to pf2 without changing P:
  Qc = P (tan phi1 - tan phi2),   and for a capacitor C across the load   Qc = 2 pi f C V^2
  so   C = Qc / (2 pi f V^2).
Check by hand for 100 kW, pf 0.70 to 0.95: tan phi1 = 1.0202, tan phi2 = 0.3287, so Qc = 69.15 kvar.

In [ ]:
def capacitor_kvar(p_watts, pf1, pf2):
    """Reactive power (var) a capacitor bank must supply to lift pf1 to pf2 at constant P."""
    return p_watts * (np.tan(np.arccos(pf1)) - np.tan(np.arccos(pf2)))


def capacitance_for(qc_var, v_rms, freq=50.0):
    """Capacitance (farad) that gives qc_var at v_rms across it: C = Qc / (2 pi f V^2)."""
    return qc_var / (2.0 * np.pi * freq * v_rms ** 2)

## Step 3: why transmission uses high voltage
For the same power, a higher voltage means a smaller current, and the line heats up with
the square of the current: loss = I^2 R. Ten times the voltage means one tenth of the
current and one hundredth of the loss.

In [ ]:
def line_loss(p_watts, v_rms, r_line, pf=1.0):
    """Power lost (W) in a line of resistance r_line (ohm) that delivers P at v_rms (single-phase)."""
    i = current_for(p_watts, v_rms, pf)
    return i * i * r_line

## Step 4: see it
Left: the power triangle before and after correction (P stays, Q shrinks, S shrinks).
Right: current needed for 100 kW as the power factor falls.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, V = 100e3, 400.0
    for pf in (0.7, 0.95):
        print(f"100 kW at {V:.0f} V, pf {pf}: I = {current_for(P, V, pf):.1f} A, Q = {reactive_for(P, pf) / 1e3:.2f} kvar")
    qc = capacitor_kvar(P, 0.7, 0.95)
    print(f"capacitor: Qc = {qc / 1e3:.2f} kvar, C = {capacitance_for(qc, V) * 1e6:.0f} microfarad")
    qd = capacitor_kvar(50e3, 0.75, 0.95)
    print(f"design: 50 kW, pf 0.75 to 0.95: Qc = {qd / 1e3:.2f} kvar, C = {capacitance_for(qd, V) * 1e6:.0f} microfarad")
    for vv in (10e3, 100e3):
        print(f"1 MW over a 5 ohm line at {vv / 1e3:.0f} kV: loss = {line_loss(1e6, vv, 5.0):.0f} W")

    fig, ax = plt.subplots(figsize=(5.5, 5))
    q1, q2 = reactive_for(P, 0.7) / 1e3, reactive_for(P, 0.95) / 1e3
    ax.plot([0, 100, 100, 0], [0, 0, q1, 0], "r-", label=f"before: pf 0.70, S = {np.hypot(100, q1):.0f} kVA")
    ax.plot([0, 100, 0], [0, q2, 0], "g-", label=f"after: pf 0.95, S = {np.hypot(100, q2):.0f} kVA")
    ax.plot([100, 100], [q2, q1], "b:", lw=3, label=f"capacitor Qc = {q1 - q2:.1f} kvar")
    ax.set_xlabel("real power P (kW)")
    ax.set_ylabel("reactive power Q (kvar)")
    ax.set_title("Power triangle before and after correction")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "power_triangle.png", dpi=120)

    pfs = np.linspace(0.5, 1.0, 51)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(pfs, current_for(P, V, pfs))
    ax2.set_xlabel("power factor")
    ax2.set_ylabel("current for 100 kW at 400 V (A)")
    ax2.set_title("Low power factor needs more current")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "current_vs_pf.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Change the load to 80 kW and the target pf to 0.98. Then answer the four questions in the
README (section F) and run the checkpoint.

## Coding tasks
1. **Easy.** Use `capacitor_kvar` and `capacitance_for` for an 80 kW load at pf 0.7 raised to 0.98 on 400 V. Predict whether the capacitor is larger or smaller than for 0.95 before you run it.
2. **Stretch.** Print the line loss for 1 MW over a 5 ohm line at 20 kV, 50 kV and 100 kV, and check that each is proportional to 1 / V squared.
3. **Explain.** Explain, using the power triangle, why correcting from 0.95 to 1.0 needs less capacitor than correcting from 0.70 to 0.75 but may still be worth less money.

## Assessment
- **A. Conceptual.** Why does a low power factor waste capacity even though the reactive power does no useful work, and how does a capacitor help?
- **B. Numerical.** A 30 kW single-phase load on 230 V has pf 0.8. Find its current, its reactive power, and the current if the pf were corrected to 1.
- **C. Graph and simulation interpretation.** From the current-against-pf graph (100 kW, 400 V), what are the currents at pf 1 and pf 0.5, and what does that tell you?
- **D. Design and optimisation challenge.** Design: a 50 kW load has pf 0.75 and must reach 0.95 at 400 V, 50 Hz. Find the exact capacitor size and capacitance. A supplier only stocks a 30 kvar bank: what pf does it give? Discuss the trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    P, Q, S, pf = L.power_triangle(230.0, 10.0, 60.0)
    checks.append(("power triangle: 230 V, 10 A, phi 60 degrees gives P = 1150 W, Q = 1992 var, S = 2300 VA, pf 0.5",
                   abs(P - 1150.0) < 0.5 and abs(Q - 1991.9) < 0.5 and abs(S - 2300.0) < 1e-9 and abs(pf - 0.5) < 1e-12))
    checks.append(("S^2 = P^2 + Q^2", abs(S ** 2 - P ** 2 - Q ** 2) < 1e-6))
    i1, i2 = L.current_for(100e3, 400.0, 0.7), L.current_for(100e3, 400.0, 0.95)
    checks.append(("worked example: 100 kW at 400 V needs 357.1 A at pf 0.70 and 263.2 A at pf 0.95",
                   abs(i1 - 357.1) < 0.05 and abs(i2 - 263.2) < 0.05))
    checks.append(("unity power factor current is P / V = 250 A", abs(L.current_for(100e3, 400.0, 1.0) - 250.0) < 1e-9))
    qc = L.capacitor_kvar(100e3, 0.7, 0.95)
    checks.append(("worked example: lifting 100 kW from pf 0.70 to 0.95 needs Qc = 69.15 kvar",
                   abs(qc / 1e3 - 69.15) < 0.01))
    checks.append(("hand calculation: tan(acos 0.70) = 1.0202 and tan(acos 0.95) = 0.3287",
                   abs(np.tan(np.arccos(0.7)) - 1.0202) < 1e-4 and abs(np.tan(np.arccos(0.95)) - 0.3287) < 1e-4))
    C = L.capacitance_for(qc, 400.0, 50.0)
    checks.append(("capacitance for 69.15 kvar at 400 V, 50 Hz is 1376 microfarad", abs(C * 1e6 - 1376.0) < 1.0))
    checks.append(("a capacitor of that size gives back Qc = 2 pi f C V^2", abs(2 * np.pi * 50.0 * C * 400.0 ** 2 - qc) < 1e-6))
    # after correction the apparent power is P / 0.95 and the new current follows
    q_after = L.reactive_for(100e3, 0.95)
    checks.append(("after correction Q = 32.87 kvar and S = 105.3 kVA (so I = 263.2 A)",
                   abs(q_after / 1e3 - 32.87) < 0.01 and abs(np.hypot(100e3, q_after) / 1e3 - 105.26) < 0.01
                   and abs(np.hypot(100e3, q_after) / 400.0 - i2) < 1e-6))
    checks.append(("before correction S = 142.9 kVA", abs(np.hypot(100e3, L.reactive_for(100e3, 0.7)) / 1e3 - 142.86) < 0.01))
    qd = L.capacitor_kvar(50e3, 0.75, 0.95)
    checks.append(("design problem: 50 kW from pf 0.75 to 0.95 needs 27.66 kvar, which is 550 microfarad at 400 V",
                   abs(qd / 1e3 - 27.66) < 0.01 and abs(L.capacitance_for(qd, 400.0) * 1e6 - 550.0) < 1.0))
    checks.append(("no correction needed: pf 0.95 to 0.95 gives 0 kvar", abs(L.capacitor_kvar(50e3, 0.95, 0.95)) < 1e-9))
    l10, l100 = L.line_loss(1e6, 10e3, 5.0), L.line_loss(1e6, 100e3, 5.0)
    checks.append(("1 MW over 5 ohm: loss is 50000 W at 10 kV and 500 W at 100 kV", abs(l10 - 50000.0) < 1e-6 and abs(l100 - 500.0) < 1e-6))
    checks.append(("ten times the voltage gives exactly one hundredth of the loss", abs(l10 / l100 - 100.0) < 1e-9))
    checks.append(("line loss grows by (0.95/0.70)^2 = 1.842 when pf drops from 0.95 to 0.70",
                   abs(L.line_loss(1e6, 10e3, 5.0, 0.7) / L.line_loss(1e6, 10e3, 5.0, 0.95) - 1.842) < 1e-3))
    s3 = 100e3 / (np.sqrt(3) * 400.0 * 0.95)
    checks.append(("three-phase: P = sqrt(3) V_line I pf holds for I = P / (sqrt(3) V pf)", abs(np.sqrt(3) * 400.0 * s3 * 0.95 - 100e3) < 1e-6))
    checks.append(("Section F answer B: 30 kW at 230 V, pf 0.8 needs 163.0 A, 130.4 A at unity, and 22.5 kvar of reactive power",
                   abs(L.current_for(30e3, 230.0, 0.8) - 163.04) < 0.01 and abs(L.current_for(30e3, 230.0, 1.0) - 130.43) < 0.01
                   and abs(L.reactive_for(30e3, 0.8) / 1e3 - 22.5) < 1e-9))
    checks.append(("Section F answer C: 100 kW at 400 V needs 500 A at pf 0.5, twice the 250 A at pf 1", abs(L.current_for(100e3, 400.0, 0.5) - 500.0) < 1e-9))
    checks.append(("line currents for 1 MW: 100 A at 10 kV and 10 A at 100 kV", abs(L.current_for(1e6, 10e3, 1.0) - 100.0) < 1e-9 and abs(L.current_for(1e6, 100e3, 1.0) - 10.0) < 1e-9))
    q1 = L.reactive_for(50e3, 0.75)
    pf_30 = 50e3 / np.hypot(50e3, q1 - 30e3)
    checks.append(("Section F answer D: the 50 kW load has Q = 44.10 kvar; a standard 30 kvar bank gives pf 0.9624, under-correcting; 27.66 kvar is the exact size for 0.95",
                   abs(q1 / 1e3 - 44.10) < 0.01 and abs(pf_30 - 0.9624) < 5e-4))
    checks.append(("before correction Q = 102.0 kvar, after Q = 32.9 kvar", abs(L.reactive_for(100e3, 0.7) / 1e3 - 102.0) < 0.05 and abs(L.reactive_for(100e3, 0.95) / 1e3 - 32.9) < 0.05))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
